In [1]:
library(PATH)
library(expm)
library(ggplot2)
library(ggtree)
library(ggtreeExtra)
library(tidytree)
library(Matrix)
library(patchwork)

options(digits = 2)
library(ComplexHeatmap)
library(grid)
library(viridis)

Loading required package: Matrix


Attaching package: ‘expm’


The following object is masked from ‘package:Matrix’:

    expm


ggtree v3.10.1 For help: https://yulab-smu.top/treedata-book/

If you use the ggtree package suite in published research, please cite
the appropriate paper(s):

Guangchuang Yu, David Smith, Huachen Zhu, Yi Guan, Tommy Tsan-Yuk Lam.
ggtree: an R package for visualization and annotation of phylogenetic
trees with their covariates and other associated data. Methods in
Ecology and Evolution. 2017, 8(1):28-36. doi:10.1111/2041-210X.12628

LG Wang, TTY Lam, S Xu, Z Dai, L Zhou, T Feng, P Guo, CW Dunn, BR
Jones, T Bradley, H Zhu, Y Guan, Y Jiang, G Yu. treeio: an R package
for phylogenetic tree input and output with richly annotated and
associated data. Molecular Biology and Evolution. 2020, 37(2):599-603.
doi: 10.1093/molbev/msz240

Guangchuang Yu.  Data Integration, Manipulation and Visualization of
Phylogenetic Trees (1st edition). Chapman and Hall/CRC. 2022,
doi

In [3]:
metadata <- read.table('/syn1/liangzhen/jinhua_jilab_project/result/scRNA/cellranger/scRNA.metadata.csv',sep=',',header=T,row.names=1)
rownames(metadata) <- metadata$cellName
unique(metadata$tumor_state)

[1] 2 1 3 4 5

# BL

In [19]:
treefiles <- grep('treefile$',list.files('/syn1/liangzhen/jinhua_jilab_project/result/DNA_Amplicon/trees/merge_all_experiments/T1/final/'),value = T)

In [20]:
treefiles

[1] "C1-167.phy.treefile"   "C103-101.phy.treefile" "C12-351.phy.treefile" 
 [4] "C13-113.phy.treefile"  "C14-153.phy.treefile"  "C15-171.phy.treefile" 
 [7] "C18-196.phy.treefile"  "C19-431.phy.treefile"  "C2-152.phy.treefile"  
[10] "C20-150.phy.treefile"  "C21-217.phy.treefile"  "C22-112.phy.treefile" 
[13] "C23-101.phy.treefile"  "C26-289.phy.treefile"  "C29-285.phy.treefile" 
[16] "C3-103.phy.treefile"   "C32-247.phy.treefile"  "C34-269.phy.treefile" 
[19] "C36-119.phy.treefile"  "C37-268.phy.treefile"  "C38-160.phy.treefile" 
[22] "C4-133.phy.treefile"   "C42-223.phy.treefile"  "C43-158.phy.treefile" 
[25] "C44-207.phy.treefile"  "C45-111.phy.treefile"  "C46-101.phy.treefile" 
[28] "C47-130.phy.treefile"  "C50-194.phy.treefile"  "C51-176.phy.treefile" 
[31] "C52-146.phy.treefile"  "C53-189.phy.treefile"  "C54-158.phy.treefile" 
[34] "C55-163.phy.treefile"  "C57-172.phy.treefile"  "C59-168.phy.treefile" 
[37] "C6-103.phy.treefile"   "C60-171.phy.treefile"  "C61-114.phy.treefile" 
[40] "C62-133.phy.treefile"  "C63-142.phy.treefile"  "C64-171.phy.treefile" 
[43] "C65-167.phy.treefile"  "C66-169.phy.treefile"  "C69-134.phy.treefile" 
[46] "C7-1176.phy.treefile"  "C70-105.phy.treefile"  "C72-114.phy.treefile" 
[49] "C73-127.phy.treefile"  "C75-135.phy.treefile"  "C79-130.phy.treefile" 
[52] "C8-220.phy.treefile"   "C80-116.phy.treefile"  "C81-120.phy.treefile" 
[55] "C82-104.phy.treefile"  "C83-114.phy.treefile"  "C84-118.phy.treefile" 
[58] "C89-114.phy.treefile"  "C9-128.phy.treefile"   "C90-114.phy.treefile" 
[61] "C95-107.phy.treefile"  "C96-105.phy.treefile"  "C97-103.phy.treefile" 
[64] "C98-105.phy.treefile"  "C99-105.phy.treefile"

In [21]:
single_state_clones <- c('C4','C16','C37','C46','C64','C69','C72','C74','C84','C113','C115','C117','C121','C125','C145','C146','C156','C165')

In [22]:
pdf('/syn1/liangzhen/jinhua_jilab_project/result/Figures/Figure3/BL_state_transition_heatmap.pdf',width=6,height=5)
for (treefile in treefiles){
    clone_size <- as.numeric(gsub('.treefile','',unlist(strsplit(treefile,split = '-'))[2]))

    if(unlist(strsplit(treefile,split = '-'))[1] %in% single_state_clones){
        next
    }
    file <- treefile
    lineageGrp <- unlist(strsplit(treefile,split = '-'))[1]
    tree <- read.tree(paste0('/syn1/liangzhen/jinhua_jilab_project/result/DNA_Amplicon/trees/merge_all_experiments/T1/final/',file))
    tree <- root(tree, outgroup = "synthetic", resolve.root = TRUE)
    #tree$edge.length <- rep(1,length(tree$edge.length)) # node length whether ==1?
    tree <- drop.tip(tree,tree$tip.label[!tree$tip.label %in% metadata$cellName])
    tree$states <- paste0(as.character(metadata[tree$tip.label,'seurat_clusters_rename']))
    df <- as.data.frame(table(tree$states))
    df$ratio <- df$Freq / sum(df$Freq)
    clusters <- as.character(df[df$ratio>0.1,'Var1'])
    tree <- drop.tip(tree,tree$tip.label[!tree$states %in% clusters])
    tree$states <- paste0(as.character(metadata[tree$tip.label,'seurat_clusters_rename']))
    tree$edge.length <- rep(1,length(tree$edge.length))
    #print(length(unique(tree$states)))

    if (length(unique(tree$states))!=1){
    #print(tree)
    Pinf <- PATH_inf(tree = tree, cell_states = "states", nstates = length(unique(tree$states)))

    options(repr.plot.width=6, repr.plot.height=5)
    plot <- Heatmap(Pinf$P,column_title = lineageGrp,rocket(256),
            row_names_gp = gpar(fontsize = 18, fontface = "bold"), 
            column_names_gp = gpar(fontsize = 18, fontface = "bold"),
            cluster_rows = F,cluster_columns = F,row_names_side = 'left',column_names_side = 'top',column_names_rot = 0,
            cell_fun = function(j, i, x, y, width, height, fill) {
                    grid.text(sprintf("%.2f", as.matrix(Pinf$P)[i, j]), x, y, gp = gpar(fontsize = 15,col = "white"))
                    },
            heatmap_legend_param = list(title = "state transitions")
                   )

    print(plot)
    write.table(Pinf$P,file=paste0('/syn1/liangzhen/jinhua_jilab_project/result/DNA_Amplicon/cell_transition/BL/',lineageGrp,'_state_transition.csv'),sep=',',quote=F,col.names = NA) 
    grid.newpage() 
}
}
dev.off()

Warning message in eval(expr, envir, enclos):
“NAs introduced by coercion”
Warning message in eval(expr, envir, enclos):
“NAs introduced by coercion”
Warning message in eval(expr, envir, enclos):
“NAs introduced by coercion”
Warning message in eval(expr, envir, enclos):
“NAs introduced by coercion”
Warning message in eval(expr, envir, enclos):
“NAs introduced by coercion”
Warning message in eval(expr, envir, enclos):
“NAs introduced by coercion”
Warning message in eval(expr, envir, enclos):
“NAs introduced by coercion”
Warning message in eval(expr, envir, enclos):
“NAs introduced by coercion”
Warning message in eval(expr, envir, enclos):
“NAs introduced by coercion”
Warning message in eval(expr, envir, enclos):
“NAs introduced by coercion”
Warning message in eval(expr, envir, enclos):
“NAs introduced by coercion”
Warning message in eval(expr, envir, enclos):
“NAs introduced by coercion”
Warning message in eval(expr, envir, enclos):
“NAs introduced by coercion”
Warning message in eval(e

png 
  2

# EE

In [23]:
treefiles <- grep('treefile$',list.files('/syn1/liangzhen/jinhua_jilab_project/result/DNA_Amplicon/trees/merge_all_experiments/T2/final/'),value = T)

In [24]:
treefiles

[1] "C1-2815.phy.treefile" "C10-173.phy.treefile" "C12-295.phy.treefile"
 [4] "C13-425.phy.treefile" "C14-231.phy.treefile" "C15-110.phy.treefile"
 [7] "C16-170.phy.treefile" "C17-134.phy.treefile" "C18-245.phy.treefile"
[10] "C2-1286.phy.treefile" "C20-104.phy.treefile" "C22-222.phy.treefile"
[13] "C23-110.phy.treefile" "C25-113.phy.treefile" "C27-124.phy.treefile"
[16] "C3-900.phy.treefile"  "C30-158.phy.treefile" "C33-130.phy.treefile"
[19] "C39-168.phy.treefile" "C49-101.phy.treefile" "C5-344.phy.treefile" 
[22] "C6-728.phy.treefile"  "C8-468.phy.treefile"  "C9-541.phy.treefile"

In [25]:
pdf('/syn1/liangzhen/jinhua_jilab_project/result/Figures/Figure3/EE_state_transition_heatmap.pdf',width=6,height=5)

for (treefile in treefiles){
    if(unlist(strsplit(treefile,split = '-'))[1] %in% single_state_clones){
        next
    }
    lineageGrp <- unlist(strsplit(treefile,split = '-'))[1]
    tree <- read.tree(paste0('/syn1/liangzhen/jinhua_jilab_project/result/DNA_Amplicon/trees/merge_all_experiments/T2/final/',treefile))
    tree <- root(tree, outgroup = "synthetic", resolve.root = TRUE)
    #tree$edge.length <- rep(1,length(tree$edge.length)) # node length whether ==1?
    tree <- drop.tip(tree,tree$tip.label[!tree$tip.label %in% metadata$cellName])
    tree$states <- paste0(as.character(metadata[tree$tip.label,'seurat_clusters_rename']))
    df <- as.data.frame(table(tree$states))
    df$ratio <- df$Freq / sum(df$Freq)
    clusters <- as.character(df[df$ratio>0.01,'Var1'])
    tree <- drop.tip(tree,tree$tip.label[!tree$states %in% clusters])
    tree$states <- paste0(as.character(metadata[tree$tip.label,'seurat_clusters_rename']))
    tree$edge.length <- rep(1,length(tree$edge.length))
    #print(length(unique(tree$states)))

    if (length(unique(tree$states))!=1){
    #print(tree)
    Pinf <- PATH_inf(tree = tree, cell_states = "states", nstates = length(unique(tree$states)))

    options(repr.plot.width=6, repr.plot.height=5)
    plot <- Heatmap(Pinf$P,column_title = lineageGrp,rocket(256),
            row_names_gp = gpar(fontsize = 18, fontface = "bold"), 
            column_names_gp = gpar(fontsize = 18, fontface = "bold"),
            cluster_rows = F,cluster_columns = F,row_names_side = 'left',column_names_side = 'top',column_names_rot = 0,
            cell_fun = function(j, i, x, y, width, height, fill) {
                    grid.text(sprintf("%.2f", as.matrix(Pinf$P)[i, j]), x, y, gp = gpar(fontsize = 15,col = "white"))
                    },
            heatmap_legend_param = list(title = "state transitions")
                   )

    print(plot)
    write.table(Pinf$P,file=paste0('/syn1/liangzhen/jinhua_jilab_project/result/DNA_Amplicon/cell_transition/EE/',lineageGrp,'_state_transition.csv'),sep=',',quote=F,col.names = NA)   
    grid.newpage() 
    }
}
dev.off()

png 
  2

# LE

In [26]:
treefiles <- grep('treefile$',list.files('/syn1/liangzhen/jinhua_jilab_project/result/DNA_Amplicon/trees/merge_all_experiments/T3/final/'),value = T)

In [27]:
treefiles

[1] "C1-1580.phy.treefile" "C10-719.phy.treefile" "C11-691.phy.treefile"
 [4] "C12-111.phy.treefile" "C13-212.phy.treefile" "C14-347.phy.treefile"
 [7] "C15-431.phy.treefile" "C16-404.phy.treefile" "C17-471.phy.treefile"
[10] "C18-156.phy.treefile" "C2-2012.phy.treefile" "C20-167.phy.treefile"
[13] "C21-110.phy.treefile" "C23-189.phy.treefile" "C24-298.phy.treefile"
[16] "C25-262.phy.treefile" "C27-195.phy.treefile" "C28-261.phy.treefile"
[19] "C3-1405.phy.treefile" "C31-175.phy.treefile" "C33-104.phy.treefile"
[22] "C35-198.phy.treefile" "C4-1581.phy.treefile" "C40-127.phy.treefile"
[25] "C41-184.phy.treefile" "C48-105.phy.treefile" "C5-1113.phy.treefile"
[28] "C58-138.phy.treefile" "C6-522.phy.treefile"  "C8-326.phy.treefile" 
[31] "C9-326.phy.treefile"

In [28]:
pdf('/syn1/liangzhen/jinhua_jilab_project/result/Figures/Figure3/LE_state_transition_heatmap.pdf',width=6,height=5)
for (treefile in treefiles){
    if(unlist(strsplit(treefile,split = '-'))[1] %in% single_state_clones){
        next
    }
    
    lineageGrp <- unlist(strsplit(treefile,split = '-'))[1]
    tree <- read.tree(paste0('/syn1/liangzhen/jinhua_jilab_project/result/DNA_Amplicon/trees/merge_all_experiments/T3/final/',treefile))
    tree <- root(tree, outgroup = "synthetic", resolve.root = TRUE)
    #tree$edge.length <- rep(1,length(tree$edge.length)) # node length whether ==1?
    tree <- drop.tip(tree,tree$tip.label[!tree$tip.label %in% metadata$cellName])
    tree$states <- paste0(as.character(metadata[tree$tip.label,'seurat_clusters_rename']))
    df <- as.data.frame(table(tree$states))
    df$ratio <- df$Freq / sum(df$Freq)
    clusters <- as.character(df[df$ratio>0.01,'Var1'])
    tree <- drop.tip(tree,tree$tip.label[!tree$states %in% clusters])
    tree$states <- paste0(as.character(metadata[tree$tip.label,'seurat_clusters_rename']))
    tree$edge.length <- rep(1,length(tree$edge.length))
    #print(length(unique(tree$states)))

    if (length(unique(tree$states))!=1){
    #print(tree)
    Pinf <- PATH_inf(tree = tree, cell_states = "states", nstates = length(unique(tree$states)))

    options(repr.plot.width=6, repr.plot.height=5)
    plot <- Heatmap(Pinf$P,column_title = lineageGrp,rocket(256),
            row_names_gp = gpar(fontsize = 18, fontface = "bold"), 
            column_names_gp = gpar(fontsize = 18, fontface = "bold"),
            cluster_rows = F,cluster_columns = F,row_names_side = 'left',column_names_side = 'top',column_names_rot = 0,
            cell_fun = function(j, i, x, y, width, height, fill) {
                    grid.text(sprintf("%.2f", as.matrix(Pinf$P)[i, j]), x, y, gp = gpar(fontsize = 15,col = "white"))
                    },
            heatmap_legend_param = list(title = "state transitions")
                   )

    print(plot)
    write.table(Pinf$P,file=paste0('/syn1/liangzhen/jinhua_jilab_project/result/DNA_Amplicon/cell_transition/LE/',lineageGrp,'_state_transition.csv'),sep=',',quote=F,col.names = NA) 
    grid.newpage() 
}
}
dev.off()

png 
  2